In [ ]:
# Cell 1 — Imports and configuration (unchanged from original)
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'evaluation' / 'test_questions.json').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline
from hybrid_rag.pipeline     import HybridRAGPipeline
from evaluation.evaluator    import run_evaluation, print_summary
from evaluation.ragas_evaluator import run_ragas_evaluation, merge_results, print_ragas_summary

RESULTS_DIR    = REPO_ROOT / 'evaluation' / 'results'
QUESTIONS_PATH = REPO_ROOT / 'evaluation' / 'test_questions.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER  = ['vector', 'vectorless', 'hybrid']
METHOD_LABELS = {'vector': 'Vector RAG', 'vectorless': 'Vectorless RAG', 'hybrid': 'Hybrid RAG'}
METHOD_COLORS = {'vector': '#1f77b4', 'vectorless': '#ff7f0e', 'hybrid': '#2ca02c'}
RAGAS_METRIC_LABELS = {
    'answer_relevancy'    : 'Answer Relevancy',
    'faithfulness'        : 'Faithfulness',
    'context_precision'   : 'Contextual Precision',
    'context_recall'      : 'Contextual Recall',
    'contextual_relevancy': 'Contextual Relevancy\n(≈ context_precision)',
}

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 160,
                     'axes.titlesize': 13, 'axes.labelsize': 11})

def save_figure(fig, filename):
    out = RESULTS_DIR / filename
    fig.savefig(out, dpi=160, bbox_inches='tight')
    return out

print('Imports ready.')

In [ ]:
# Cell 2 — Initialise pipelines
with open(QUESTIONS_PATH) as f:
    _q = json.load(f)
print(f'Loaded {len(_q["questions"])} questions for the benchmark.')

print('Initialising pipelines...')
vec    = VectorRAGPipeline()
vl     = VectorlessRAGPipeline()
hybrid = HybridRAGPipeline()
print('All three pipelines are ready.')